# Parallel Processing

This notebook demonstrates parallel processing capabilities for batch operations:
- Batch parsing of multiple structures
- Parallel connectivity list generation
- Performance comparisons
- Different backends (multiprocessing, threading, sequential)


In [ ]:
from rna_secstruct.parallel import batch_parse, batch_connectivity, batch_apply
from rna_secstruct import SecStruct
import time


## Batch Parsing


In [ ]:
# Create multiple structures
sequences = [
    "GGGAAACCC",
    "AAAGGGCCC",
    "CCCGAAAGGG",
    "GGGAAACCC",
    "AAAGGGCCC"
]
structures = [
    "(((...)))",
    "(((...)))",
    "(((...)))",
    "(((...)))",
    "(((...)))"
]

# Parse sequentially
start = time.time()
parsed_seq = batch_parse(sequences, structures, n_jobs=1, backend="sequential")
time_seq = time.time() - start

print(f"Parsed {len(parsed_seq)} structures sequentially in {time_seq:.4f}s")
print(f"First structure: {parsed_seq[0].sequence}")


## Performance Comparison


In [ ]:
# Create larger dataset for performance testing
n_structures = 20
sequences = ["GGGAAACCC"] * n_structures
structures = ["(((...)))"] * n_structures

# Sequential
start = time.time()
result_seq = batch_parse(sequences, structures, n_jobs=1, backend="sequential")
time_seq = time.time() - start

# Threading (if available)
try:
    start = time.time()
    result_thread = batch_parse(sequences, structures, n_jobs=2, backend="threading")
    time_thread = time.time() - start
    print(f"Sequential: {time_seq:.4f}s")
    print(f"Threading (2 jobs): {time_thread:.4f}s")
    print(f"Speedup: {time_seq/time_thread:.2f}x")
except Exception as e:
    print(f"Threading not available: {e}")

print(f"\nAll results match: {all(r.sequence == result_seq[0].sequence for r in result_seq)}")


## Batch Connectivity Lists


In [ ]:
sequences = ["GGGAAACCC", "AAAGGGCCC", "CCCGAAAGGG"]
structures = ["(((...)))", "(((...)))", "(((...)))"]

# Generate connectivity lists in parallel
conn_lists = batch_connectivity(
    sequences, structures, n_jobs=1, backend="sequential"
)

print(f"Generated {len(conn_lists)} connectivity lists")
for i, conn in enumerate(conn_lists):
    if hasattr(conn, 'connections'):
        print(f"  Structure {i}: {conn.connections[:5]}...")
    else:
        print(f"  Structure {i}: {conn[:5]}...")


## Batch Apply Operations


In [ ]:
# Create structures
structs = [
    SecStruct("GGGAAACCC", "(((...)))"),
    SecStruct("AAAGGGCCC", "(((...)))"),
    SecStruct("CCCGAAAGGG", "(((...)))")
]

# Apply function to all structures
num_bp = batch_apply(
    structs,
    lambda s: s.get_num_basepairs(),
    n_jobs=1,
    backend="sequential"
)

print(f"Base pair counts: {num_bp}")

# Apply multiple functions
gc_contents = batch_apply(
    structs,
    lambda s: s.get_gc_content(),
    n_jobs=1,
    backend="sequential"
)
print(f"GC contents: {[f'{gc:.2%}' for gc in gc_contents]}")


## Error Handling


In [ ]:
# Test with mismatched lengths
try:
    batch_parse(["GGGAAACCC"], ["(((...)))", "(((...)))"])
except ValueError as e:
    print(f"Caught expected error: {e}")

# Test with invalid backend
try:
    batch_parse(["GGGAAACCC"], ["(((...)))"], backend="invalid")
except ValueError as e:
    print(f"\nCaught expected error: {e}")


## Integration with Pandas


In [ ]:
try:
    import pandas as pd
    
    # Create DataFrame
df = pd.DataFrame({
        'sequence': ['GGGAAACCC'] * 10,
        'structure': ['(((...)))'] * 10
    })
    
    # Use batch_parse for efficient processing
    sequences = df['sequence'].tolist()
    structures = df['structure'].tolist()
    
    start = time.time()
    structs = batch_parse(sequences, structures, n_jobs=1, backend="sequential")
    time_batch = time.time() - start
    
    df['secstruct'] = structs
    
    print(f"Processed {len(df)} structures in {time_batch:.4f}s")
    print(f"DataFrame shape: {df.shape}")
except ImportError:
    print("Pandas not available")
